In [1]:
import pandas as pd
import sys
from pandas import read_csv
import matplotlib.pyplot as plt
print("Python version:", sys.version)
print("Pandas version:", pd.__version__)
import numpy as np
from pathlib import Path
import yfinance as yf

Python version: 3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]
Pandas version: 3.0.1


In [3]:
import unicodedata


def normalize_text(text):
    text = str(text).strip()
    text = text.replace('Đ', 'D').replace('đ', 'd')
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    return text.lower()


def rename_columns(df, column_aliases=None):
    # Mac dinh cho cac bo du lieu co cau truc giong VN_index.csv
    default_aliases = {
        'ngay': 'time',
        'lan cuoi': 'close',
        'mo': 'open',
        'cao': 'high',
        'thap': 'low',
        'kl': 'volume',
        '% thay doi': 'pct_change',
    }

    aliases = default_aliases.copy()
    if column_aliases:
        aliases.update({normalize_text(k): v for k, v in column_aliases.items()})

    rename_map = {}
    for col in df.columns:
        normalized_col = normalize_text(col)
        if normalized_col in aliases:
            rename_map[col] = aliases[normalized_col]

    return df.rename(columns=rename_map)


def parse_number(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)

    text = str(value).strip().replace(',', '')
    if text == '':
        return np.nan
    return float(text)


def convert_to_datetime(df, time_col='time', date_format='%d/%m/%Y'):
    df = df.copy()
    if time_col in df.columns:
        df[time_col] = pd.to_datetime(df[time_col], format=date_format, errors='coerce')
    return df


def preprocess_volume(df, volume_col='volume'):
    def convert_volume(volume):
        if pd.isna(volume):
            return np.nan
        if isinstance(volume, (int, float, np.number)):
            return float(volume)

        text = str(volume).strip().replace(',', '')
        if text == '':
            return np.nan

        multiplier = {'K': 1e3, 'M': 1e6, 'B': 1e9}
        suffix = text[-1].upper()
        if suffix in multiplier:
            return float(text[:-1]) * multiplier[suffix]
        return float(text)

    df = df.copy()
    if volume_col in df.columns:
        df[volume_col] = df[volume_col].apply(convert_volume)
    return df


def preprocess_numeric_columns(df, numeric_cols=('close', 'open', 'high', 'low')):
    df = df.copy()
    for col in numeric_cols:
        if col in df.columns:
            df[col] = df[col].apply(parse_number)
    return df


def preprocess_pct_change(df, pct_col='pct_change'):
    df = df.copy()
    if pct_col in df.columns:
        cleaned = (
            df[pct_col]
            .astype(str)
            .str.replace('%', '', regex=False)
            .str.replace(',', '', regex=False)
            .str.strip()
            .replace({'': np.nan})
        )
        df[pct_col] = cleaned.astype(float)
    return df


def preprocess_market_dataframe(
    df,
    column_aliases=None,
    numeric_cols=('close', 'open', 'high', 'low'),
    time_col='time',
    volume_col='volume',
    pct_col='pct_change',
    date_format='%d/%m/%Y'
    ):
    df = rename_columns(df, column_aliases=column_aliases)
    df = convert_to_datetime(df, time_col=time_col, date_format=date_format)
    df = preprocess_numeric_columns(df, numeric_cols=numeric_cols)
    df = preprocess_volume(df, volume_col=volume_col)
    df = preprocess_pct_change(df, pct_col=pct_col)
    if time_col in df.columns:
        df = df.sort_values(time_col).reset_index(drop=True)
    return df

In [5]:
#download stock data using yfinance
def get_stock_data(symbol, start="2008-01-01", end="2026-01-01"):
    """
    Downloads daily stock data for a given symbol and date range.
    """
    try:
        # Download data with 1-day interval
        data = yf.download(
            symbol, 
            start=start, 
            end=end, 
            interval="1d", 
            auto_adjust=True
        )
        
        if data.empty:
            print(f"No data found for {symbol}.")
            return pd.DataFrame()
            
        return data
        
    except Exception as e:
        print(f"An error occurred: {e}")
        return pd.DataFrame()

- dax_40, snp 500, topix:  không có volume
 -> tai lai tu yfinance
- euro next 100, ibex 35, smi: xu ly lai cot volumn
-> viet ham xu ly
- vn_index, vn30: tinh lai return 1 day
-> viet ham xu ly

In [4]:

# Tìm kiếm theo tên công ty hoặc từ khóa
search = yf.Search("vn30", max_results=10)
# Hiển thị danh sách các mã (quotes) tìm được
for quote in search.quotes:
    print(f"Symbol: {quote['symbol']} - Name: {quote['shortname']}")


Symbol: FUEMAV30.VN - Name: MIRAE ASSET (VIETNAM) FM LTD
Symbol: FUESSV30.VN - Name: SSIAM
Symbol: 245710.KS - Name: ACE VietnamVN30(Synth)
Symbol: 570063.KS - Name: KIS Vietnam VN30 Futures ETN(H)
Symbol: 570064.KS - Name: KIS Inverse Vietnam VN30 Future
Symbol: 570065.KS - Name: KIS Leverage VN30 Futures ETN(H
Symbol: 570066.KS - Name: KIS Inverse 2X VN30 Futures ETN


dax_40 -> ^GDAXI

s&p500 -> ^GSPC

topix -> Nikkei 225 (^N225) vi ko tải được dữ liệu của topic

-> tiến hành tải dữ liệu 

In [ ]:
dax_40_path = "D:\\UIT\\1003_EPA-Project_UIT\\dataset\\DAX_40.csv"
snp_500_path = "D:\\UIT\\1003_EPA-Project_UIT\\dataset\\SP500.csv"
nikkei_225_path = "D:\\UIT\\1003_EPA-Project_UIT\\dataset\\Nikkei_225.csv"

In [ ]:
# array of stock symbols
stock_symbols = ["^GDAXI", "^GSPC", "^N225"]
#loop to download data for each stock symbol
stock_data = {}
for symbol in stock_symbols:
    stock_data[symbol] = get_stock_data(symbol,start="2008-01-01", end="2026-01-01")

data_paths = {
     "^GDAXI": Path(r"D:\\UIT\\1003_EPA-Project_UIT\\dataset\\raw\\DAX_40.csv"),
     "^GSPC": Path(r"D:\\UIT\\1003_EPA-Project_UIT\\dataset\\raw\\snp500.csv"),
    "^N225": Path(r"D:\\UIT\\1003_EPA-Project_UIT\\dataset\\raw\\Nikkei_225.csv")
    }

# save as csv files for each stock symbol
saved_data = {}
for symbol, data in stock_data.items():
    if not data.empty:
        path = data_paths.get(symbol)
        if path:
            data.to_csv(path, index=True)
            saved_data[symbol] = path
            print(f"Data for {symbol} saved to {path}")
        else:
            print(f"No path defined for {symbol}, skipping save.")
    else:
        print(f"No data to save for {symbol}.")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path


def load_raw_index_csv(path):
    """Load CSV that can be single-header or Yahoo-style 3-line multi-header."""
    csv_path = Path(path)
    if not csv_path.exists():
        raise FileNotFoundError(f"File not found: {csv_path}")

    # Detect Yahoo multi-header format by checking first column names.
    probe = pd.read_csv(csv_path, nrows=3, header=None)
    first_col = probe.iloc[:, 0].astype(str).str.strip().str.lower().tolist()

    if first_col[:3] == ["price", "ticker", "date"]:
        df = pd.read_csv(csv_path, skiprows=[1, 2])
        if "Price" in df.columns:
            df = df.rename(columns={"Price": "Date"})
    else:
        df = pd.read_csv(csv_path)

    if df.empty:
        raise ValueError(f"Input file is empty: {csv_path}")

    return df


def normalize_ohlcv_schema(df):
    """Normalize dataframe to KOSPI-like schema columns."""
    out = df.copy()
    out.columns = [str(c).strip().lower().replace(" ", "_") for c in out.columns]

    column_map = {
        "date": "time",
        "time": "time",
        "close": "close",
        "open": "open",
        "high": "high",
        "low": "low",
        "volume": "volume",
    }

    selected = {}
    for src, dst in column_map.items():
        if src in out.columns:
            selected[dst] = out[src]

    required = ["time", "close", "open", "high", "low", "volume"]
    missing = [c for c in required if c not in selected]
    if missing:
        raise ValueError(f"Missing required columns after mapping: {missing}")

    norm = pd.DataFrame(selected)

    norm["time"] = pd.to_datetime(norm["time"], format="%Y-%m-%d", errors="coerce")
    norm = norm.dropna(subset=["time"]).sort_values("time").reset_index(drop=True)

    numeric_cols = ["close", "open", "high", "low", "volume"]
    for col in numeric_cols:
        norm[col] = pd.to_numeric(norm[col], errors="coerce")

    return norm


def add_return_1_day(df):
    """Add daily log return column based on close price."""
    out = df.copy()
    out["return_1_day"] = np.log(out["close"] / out["close"].shift(1))
    return out


def convert_raw_to_kospi_schema(source_csv, target_csv):
    raw = load_raw_index_csv(source_csv)
    row_before = len(raw)

    norm = normalize_ohlcv_schema(raw)
    final_df = add_return_1_day(norm)
    final_df = final_df[["time", "close", "open", "high", "low", "volume", "return_1_day"]]

    target_path = Path(target_csv)
    target_path.parent.mkdir(parents=True, exist_ok=True)
    final_df.to_csv(target_path, index=False)

    print(f"Rows before: {row_before} | Rows after: {len(final_df)}")
    if not final_df.empty:
        print(f"Time range: {final_df['time'].min()} -> {final_df['time'].max()}")
    print("Null counts:\n", final_df.isna().sum())

    return final_df


# Sample usage
source_csv = r"D:\UIT\1003_EPA-Project_UIT\dataset\raw\DAX_40.csv"
target_csv = r"D:\UIT\1003_EPA-Project_UIT\dataset\DAX_40.csv"
dax_converted = convert_raw_to_kospi_schema(source_csv, target_csv)
dax_converted.head()

In [ ]:
# Batch conversion for other raw index files
batch_jobs = [
    (
        r"D:\\UIT\\1003_EPA-Project_UIT\\dataset\\raw\\snp500.csv",
        r"D:\\UIT\\1003_EPA-Project_UIT\\dataset\\snp500.csv",
    ),
    (
        r"D:\\UIT\\1003_EPA-Project_UIT\\dataset\\raw\\Nikkei_225.csv",
        r"D:\\UIT\\1003_EPA-Project_UIT\\dataset\\Topix_500.csv",
    ),
]

batch_results = {}
for source_path, target_path in batch_jobs:
    print("=" * 80)
    print(f"Converting: {source_path}")
    converted_df = convert_raw_to_kospi_schema(source_path, target_path)
    batch_results[target_path] = converted_df

print("=" * 80)
print("Batch conversion completed.")
list(batch_results.keys())

In [41]:
euro_next_path = r"D:\UIT\1003_EPA-Project_UIT\dataset\EuroNext_100.csv"
ibex_35_path = r"D:\UIT\1003_EPA-Project_UIT\dataset\IBEX_35.csv"
smi_path = r"D:\UIT\1003_EPA-Project_UIT\dataset\SMI.csv"

euro_next_df = read_csv(euro_next_path)
ibex_35_df = read_csv(ibex_35_path)
smi_df = read_csv(smi_path)


In [42]:
euro_next_df

,time,close,open,high,low,volume,return_1_day
0,2008-01-03,983.06,982.83,986.86,977.07,285.62M,-0.001921
1,2008-01-04,967.88,982.39,986.79,963.84,365.05M,-0.015562
2,2008-01-07,968.20,964.90,971.38,963.50,393.51M,0.000331
3,2008-01-08,971.79,972.18,978.30,971.01,366.15M,0.003701
4,2008-01-09,960.25,965.28,965.41,957.55,422.37M,-0.011946
...,...,...,...,...,...,...,...
4605,2025-12-23,1707.53,1708.06,1710.14,1702.87,163.87M,-0.000111
4606,2025-12-24,1706.76,1707.91,1710.41,1706.59,113.36M,-0.000451
4607,2025-12-29,1708.46,1707.20,1712.49,1704.20,184.13M,0.000996
4608,2025-12-30,1721.53,1708.49,1724.19,1707.19,189.41M,0.007621


In [46]:
# function to preprocess value of volume for EuroNext_100, IBEX_35, SMI datasets
def preprocess_volume(df, volume_col='volume'):
    def convert_volume(volume):
        if pd.isna(volume):
            return np.nan
        if isinstance(volume, (int, float, np.number)):
            return float(volume)

        text = str(volume).strip().replace(',', '').replace('.', '')
        if text == '':
            return np.nan

        multiplier = {'K': 1e3, 'M': 1e6, 'B': 1e9}
        suffix = text[-1].upper()
        if suffix in multiplier:
            return float(text[:-1]) * multiplier[suffix]
        return float(text)

    df = df.copy()
    if volume_col in df.columns:
        df[volume_col] = df[volume_col].apply(convert_volume)
    return df

In [47]:
euro_next_df = preprocess_volume(euro_next_df, volume_col='volume')
ibex_35_df = preprocess_volume(ibex_35_df, volume_col='volume')
smi_df = preprocess_volume(smi_df, volume_col='volume')

In [48]:
euro_next_df.to_csv(r"D:\UIT\1003_EPA-Project_UIT\dataset\EuroNext_100.csv", index=False)
ibex_35_df.to_csv(r"D:\UIT\1003_EPA-Project_UIT\dataset\IBEX_35.csv", index=False)
smi_df.to_csv(r"D:\UIT\1003_EPA-Project_UIT\dataset\SMI.csv", index=False)

In [2]:
vn30_df = read_csv(r"D:\UIT\1003_EPA-Project_UIT\dataset\VN30_index.csv")


In [6]:
# calculate return_1_day for vn30_df by using log-return = np.log(df['close'] / df['close'].shift(1))
vn30_df['return_1_day'] = np.log(vn30_df['close'] / vn30_df['close'].shift(1))

In [8]:
vn30_df.to_csv(r"D:\UIT\1003_EPA-Project_UIT\dataset\VN30_index.csv", index=False)

In [18]:
vn30_df['return_1_day'] = np.log(vn30_df['close'] / vn30_df['close'].shift(1))

In [19]:
vn30_df.to_csv(r"D:\UIT\1003_EPA-Project_UIT\dataset\VN30_index.csv", index=False)

In [ ]:
#drop nan values of return_1d in vn30_df
vn30_df = vn30_df.dropna()